In [60]:
import json
import networkx as nx
import pickle
import pandas as pd

In [61]:
def load_networkx(file):
    with open(file) as infile:
        network = json.load(infile)
    G = nx.readwrite.json_graph.cytoscape_graph(network)
    return G

def load_pickle(file):
    """
    load a pickle file into a local variable
    can be a pandas dataframe or dictionary
    """
    pickle_in = open(file, "rb")
    pickle_file = pickle.load(pickle_in)
    pickle_in.close()
    return pickle_file

In [ ]:
Gtot = load_networkx('../results_mi/multilevelnetwork_weighted_mi.json')
# for node, attrs in Gtot.nodes(data=True):
#     print(f"Node {node}: {attrs}")
# for u, v, attrs in Gtot.edges(data=True):
#     print(f"Edge {u}-{v}: {attrs}")
node_dict = load_pickle('../node_dictionary.pickle')

In [63]:
node_to_layer = {node: layer for layer, nodes in node_dict.items() for node in nodes} #inverted dict
pet_nodes = set(node_dict["PET"])
rows = []
for u, v, attrs in Gtot.edges(data=True):
    if u in pet_nodes or v in pet_nodes:
        pet = u if u in pet_nodes else v
        other = v if u in pet_nodes else u
        
        if other not in pet_nodes: #not PET-PET
            rows.append({"nodo_pet": pet, "nodo_otro": other, "layer_otro": node_to_layer.get(other, "unknown"), "weight": attrs.get("weight", None)})
df = pd.DataFrame(rows)
df.sort_values(by='weight', ascending=False, inplace=True)
df

,nodo_pet,nodo_otro,layer_otro,weight
0,MEAN_METAROI_PONSVERMIS,EUR_AB42,MOLECULAR,1.000000
4360,CINGPSTL02_FDG,R_HIPPO,MRI,1.000000
4311,ANGULR03_FDG,ST94CV,MRI,1.000000
4318,ANGULR04_FDG,CEREB_TCC,MRI,1.000000
4319,ANGULR04_FDG,RHIPPO,MRI,1.000000
...,...,...,...,...
4747,HCI_2014_FDG,HMONSET,RISKFACTORS,0.005382
2852,RH_CAUDALANTERIORCINGULATE_AV45,HMONSET,RISKFACTORS,0.005376
1701,LH_INFERIORTEMPORAL_AV45,HMONSET,RISKFACTORS,0.005189
1396,LH_CAUDALANTERIORCINGULATE_AV45,HMONSET,RISKFACTORS,0.005188


In [64]:
#mean MI by layer and count edges
summary = df.groupby("layer_otro").agg(
    mean_weight=("weight", "mean"),
    count_edges=("weight", "count")
).reset_index()

summary

,layer_otro,mean_weight,count_edges
0,MOLECULAR,0.947387,813
1,MRI,0.978623,2283
2,PHENOTYPE,0.697467,890
3,RISKFACTORS,0.165471,771


In [65]:
fdg_pet_nodes = {n for n in pet_nodes if n.endswith("_FDG")}
rows_fdg = []

for u, v, attrs in Gtot.edges(data=True):
    if u in fdg_pet_nodes or v in fdg_pet_nodes:
        pet = u if u in fdg_pet_nodes else v
        other = v if u in fdg_pet_nodes else u
        
        if other not in pet_nodes:
            rows_fdg.append({
                "FDG node": pet,
                "Node from other layer": other,
                "Layer": node_to_layer.get(other, "unknown"),
                "Mutual Information": attrs.get("weight", None)
            })

df_fdg = pd.DataFrame(rows_fdg)
df_fdg["Mutual Information"] = df_fdg["Mutual Information"].round(3)
df_fdg.sort_values(by="Mutual Information", ascending=False, inplace=True)
df_fdg.to_csv('fdgpet_connections.csv', index=False) 
df_fdg

,FDG node,Node from other layer,Layer,Mutual Information
0,ANGULR01_FDG,EUR_AB42,MOLECULAR,1.000
362,CINGPSTR12_FDG,ST35CV,MRI,1.000
319,CINGPST05_FDG,ST56CV,MRI,1.000
330,CINGPST07_FDG,ST31CV,MRI,1.000
332,CINGPST07_FDG,ST56CV,MRI,1.000
...,...,...,...,...
405,TMPINFR01_FDG,HMNEURSM,RISKFACTORS,0.011
493,TMPINFR06_FDG,HMNEURSM,RISKFACTORS,0.010
156,ANGULR02_FDG,MH15DRUG,RISKFACTORS,0.008
173,ANGULL02_FDG,MH15DRUG,RISKFACTORS,0.006


In [66]:
summary = df_fdg.groupby("Layer").agg(
    mean_weight=("Mutual Information", "mean"),
    count_edges=("Mutual Information", "count")
).reset_index()

summary

,Layer,mean_weight,count_edges
0,MOLECULAR,0.967433,60
1,MRI,0.991136,286
2,PHENOTYPE,0.778721,129
3,RISKFACTORS,0.181797,148
